In [1]:
!pip install pandas numpy scikit-learn matplotlib seaborn joblib


   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.7 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.7 MB 1.8 MB/s eta 0:00:05
   ------ --------------------------------- 1.3/8.7 MB 2.1 MB/s eta 0:00:04
   -------- ------------------------------- 1.8/8.7 MB 2.2 MB/s eta 0:00:04
   ------------ --------------------------- 2.6/8.7 MB 2.6 MB/s eta 0:00:03
   --------------- ------------------------ 3.4/8.7 MB 2.8 MB/s eta 0:00:02
   ------------------- -------------------- 4.2/8.7 MB 2.9 MB/s eta 0:00:02
   ------------------------ --------------- 5.2/8.7 MB 3.1 MB/s eta 0:00:02
   ------------------------------ --------- 6.6/8.7 MB 3.5 MB/s eta 0:00:01
   ---------------------------------- ----- 7.6/8.7 MB 3.7 MB/s eta 0:00:01
   ---------------------------------------  8.7/8.7 MB 3.8 MB/s eta 0:00:01
   ---------------------------------------- 8.7/8.7 MB 3.7 MB/s eta 0:00:00
   -----------------------


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import joblib


In [5]:
# I didn't have the dataset locally so I just grabbed it from a public link.
# It's the Pima Indians Diabetes dataset, pretty standard for ML beginners.

url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"

df = pd.read_csv(url)

# checking what the dataset looks like.
df.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [6]:
# Some columns in this dataset randomly have "0" for things
# that literally can't be zero (e.g., glucose can't be 0 unless you're dead).
# So I turned those zeros into NaN so I can fill them properly.

cols_where_zero_is_weird = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

df_clean = df.copy()
for col in cols_where_zero_is_weird:
    df_clean[col] = df_clean[col].replace(0, np.nan)

# Splitting features and target. 'Outcome' is the 0/1 diabetes label.
X = df_clean.drop("Outcome", axis=1)
y = df_clean["Outcome"]

# I made a small preprocessing pipeline:
# 1. Fill missing values with the median of each column
# 2. Scale everything so ML models behave better
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Splitting into training/testing sets (80/20 split).
# I used stratify to keep the 0/1 ratio balanced.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Fit the preprocessing stuff ONLY on training data (no cheating)
X_train_prepped = pipeline.fit_transform(X_train)

# Then apply the same transformation to test data
X_test_prepped = pipeline.transform(X_test)


In [7]:
# Trying two models:
# 1. Logistic Regression (simple baseline)
# 2. Random Forest (usually performs better on this dataset)

# Logistic Regression baseline
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_prepped, y_train)

# Random Forest with random-ish params I tested
rf_model = RandomForestClassifier(
    n_estimators=120,
    max_depth=8,
    random_state=42
)
rf_model.fit(X_train_prepped, y_train)

print("Both models trained successfully.")


Both models trained successfully.


In [8]:
# Quick helper function so I don't rewrite the same lines 20 times
def get_scores(model, X_test, y_test):
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]  # probability of being diabetic

    scores = {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1 Score": f1_score(y_test, preds),
        "ROC AUC": roc_auc_score(y_test, probs)
    }
    return scores

log_results = get_scores(log_model, X_test_prepped, y_test)
rf_results = get_scores(rf_model, X_test_prepped, y_test)

print("Logistic Regression:", log_results)
print("Random Forest:", rf_results)


Logistic Regression: {'Accuracy': 0.7077922077922078, 'Precision': 0.6, 'Recall': 0.5, 'F1 Score': 0.5454545454545454, 'ROC AUC': 0.812962962962963}
Random Forest: {'Accuracy': 0.7402597402597403, 'Precision': 0.6666666666666666, 'Recall': 0.5185185185185185, 'F1 Score': 0.5833333333333334, 'ROC AUC': 0.8105555555555556}


In [9]:
# I picked the model with the better ROC AUC score.
# In my case, RF usually wins, but I still checked.

best_model = rf_model if rf_results["ROC AUC"] > log_results["ROC AUC"] else log_model

# Saving the chosen model and the preprocessing pipeline
joblib.dump(best_model, "diabetes_model.joblib")
joblib.dump(pipeline, "preprocessing_pipeline.joblib")

print("Model saved as diabetes_model.joblib")


Model saved as diabetes_model.joblib


In [10]:
# Trying a random sample to see if the model actually predicts something reasonable.

# This is just a made-up example patient.
example_person = np.array([[2, 120, 70, 28, 80, 32.5, 0.45, 35]])

# Apply the same preprocessing pipeline
example_prepped = pipeline.transform(example_person)

# Predict
pred = best_model.predict(example_prepped)[0]
prob = best_model.predict_proba(example_prepped)[0, 1]

print("Prediction (1 = diabetic):", pred)
print("Probability:", prob)


Prediction (1 = diabetic): 0
Probability: 0.2523087671691985


c:\Users\samri\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
